_Python_ package [neurolab](https://github.com/zueve/neurolab) provides a working environment for ANN.

If you don't have it installed, it can be installed with `pip`, just executing the following cell. Besides, neurolab needs numpy < 2.0, so we install it too. If installing such a numpy version in your computer presents any problem, you might create a virtual environment (`python -m venv venv_name`)

In [ ]:
!pip install neurolab
!pip install numpy==1.26.4

# Pandas and sklearn (scikit-learn) are required too
%pip install pandas
%pip install scikit-learn 

# Perceptrons

We can implement a neural network having a single layer of perceptrons (apart from input units) using _neurolab_ package as an instance of the class `newp`. In order to do so we need to provide the following parameters:
* `minmax`: a list with the same length as the number of input neurons. The $i$-th element on this list is a list of two numbers, indicating the range of input values for the $i$-th neuron.
* `cn`: number of output neurons.
* `transf`: activation function (default value is threshold).

Therefore, when we choose 1 as the value of parameter `cn`, we will be representing a simple perceptron having as many inputs as the length of the list associated to `minmax`.

Let us start by creating a simple perceptron with two inputs, both of them ranging in $[0, 1]$, and with threshold activation function.

In [ ]:
from neurolab import net

perceptron = net.newp(minmax=[[0, 1], [0, 1]], cn=1)

The instance that we just created has the following attributes:
* `inp_minmax`: range of input values.
* `co`: number of output neurons.
* `trainf`: training function (the only one specific for single-layer perceptrons is the Delta rule).
* `errorf`: error function (default value is half of SSE, _sum of squared errors_)

The layers of the neural network (input layer does not count, thus in our example there is only one) are stored in a list associated with the attribute `layers`. Each layer is an instance of the class `Layer` and has the following attributes:
* `ci`: number of inputs.
* `cn`: number of neurons on it.
* `co`: number of outputs.
* `np`: dictionary with an element `'b'` that stores an array with the neurons' biasses (terms $a_0 w_0$, default value is 0) and an element `'w'` that stores an array with the weights associated with the incoming connections arriving on each neuron (default value is 0).

In [ ]:
print('Perceptron:')
print(f'No. of output neurons: {perceptron.co}')
print(f'Training function/method: {perceptron.trainf}')
print(f'Error function: {perceptron.errorf}')
print('---')

# There is only one layer
layer = perceptron.layers[0]
print('\nLayer 0 weights')
print(layer.np)

Next, let us train the perceptron so that it models the logic gate _and_.

First of all, let us define the training set. We shall do it indicating on one hand an array or list of lists with the input values corresponding to the examples, and on the other hand a different array or list of lists with the expected ouput for each example.

In [ ]:
import numpy as np

input_values = np.array([[0, 0],
                         [0, 1],
                         [1, 0],
                         [1, 1]])

expected_outcomes = np.array([[0],
                              [0],
                              [0],
                              [1]])

The method `step` allows us to calculate the output of the neural network for a single example, and the method `sim` for all the examples.

In [ ]:
perceptron.step([1, 1])

In [ ]:
perceptron.sim(input_values)

Let us check which is the initial error of the perceptron, before the training.

__Important__: the arguments of the error function must be arrays.

In [ ]:
perceptron.errorf(expected_outcomes, perceptron.sim(input_values))

Let us next proceed to train the perceptron. We shall check that, as expected (since the training set is linearly separable), we are able to decrease the value of the error down to zero.

__Note__: the method `train` that runs the training algorithm on the neural network returns a list showing the value of the network error after each of the _epochs_. More precisely, an epoch represents the set of operations performed by the training algorithm until all the examples  of the training set have been considered.

In [ ]:
perceptron.train(input_values, expected_outcomes)

In [ ]:
print(f'Perceptron weights after training: {perceptron.layers[0].np}')
print(f'Error: {perceptron.errorf(expected_outcomes, perceptron.sim(input_values))}')

# Feed forward perceptrons

Package _neurolab_ implements a feed forward artificial neural network as an instance of the class `newff`. In order to do so, we need to provide the following parameters:
* `minmax`: a list with the same length as the number of input neurons. The $i$-th element on this list is a list of two numbers, indicating the range of input values for the $i$-th neuron.
* `size`: a list with the same length as the number of layers (except the input layer). The $i$-th element on this list is a number, indicating the number of neurons for the $i$-th layer.
* `transf`: a list with the same length as the number of layers (except the input layer). The $i$-th element on this list is the activation function (default value is [hyperbolic tangent](https://en.wikipedia.org/wiki/Hyperbolic_functions) for the neurons of the $i$-th layer.

Next, let us create a neural network with two inputs ranging over $[0, 1]$, one hidden layer having two neurons and an output layer with only one neuron. All neurons should have the sigmoid function as activation function (you may look for further available activation functions at https://pythonhosted.org/neurolab/lib.html#module-neurolab.trans).

In [ ]:
from neurolab import trans

my_net = net.newff(minmax=[[0, 1], [0, 1]],
                   size=[2, 1],
                   transf=[trans.LogSig()]*2)

The instance that we just created has the following attributes:
* `inp_minmax`: range of input values.
* `co`: number of output neurons.
* `trainf`: training function (default value is [Broyden–Fletcher–Goldfarb–Shanno algorithm](https://en.wikipedia.org/wiki/Broyden%E2%80%93Fletcher%E2%80%93Goldfarb%E2%80%93Shanno_algorithm)).
* `errorf`: error function (default value is half of SSE, _sum of squared errors_)

The layers of the neural network (input layer excluded) are stored in a list associated with the attribute `layers`. Each layer is an instance of the class `Layer` and has the following attributes:
* `ci`: number of inputs.
* `cn`: number of neurons on it.
* `co`: number of outputs.
* `np`: dictionary with an element `'b'` that stores an array with the neurons' biasses (terms $a_0 w_0$) and an element `'w'` that stores an array with the weights associated with the incoming connections arriving on each neuron. Default values for the biasses and the weights are calculated following the [Nguyen-Widrow initialization algorithm](https://web.stanford.edu/class/ee373b/nninitialization.pdf).

In [ ]:
print('Network:')
print(f'No. of output neurons: {my_net.co}')
print(f'Training function/method: {my_net.trainf}')
print(f'Error function: {my_net.errorf}')
print('---')

# Hidden layer is layer 0
hidden_layer = my_net.layers[0]
print(f'\nhl weights: {hidden_layer.np}')

# Output layer is layer 1 in this case (or -1 in general)
output_layer = my_net.layers[-1]
print(f'\nol weights: {output_layer.np}')

It is possible to modify the initialization of the biases and weights, you may find available initialization options at https://pythonhosted.org/neurolab/lib.html#module-neurolab.init.<br>
Let us for example set all of them to zero, using the following instructions:

In [ ]:
from neurolab import init

for l in my_net.layers:
    l.initf = init.init_zeros
my_net.init()
print(hidden_layer.np)
print(output_layer.np)

It is also possible to modify the training algorithm, you may find available implemented options at https://pythonhosted.org/neurolab/lib.html#module-neurolab.train.<br>
Let us for example switch to the _gradient descent backpropagation_, using the following instructions:

In [ ]:
from neurolab import train

my_net.trainf = train.train_gd

Finally, we can also modify the error function to be used when training, you may find available options at https://pythonhosted.org/neurolab/lib.html#module-neurolab.error.<br>
Let us for example choose the _mean squared error_, using the following instructions:

In [ ]:
from neurolab import error

my_net.errorf = error.MSE()

Next, let us train our neural network so that it models the behaviour of the _xor_ logic gate.

First, we need to split our training set into two components: on one hand an array or a list of lists with the input data corresponding to each example, *xor_in* , and on the other hand an array or list of lists with the correct expected ouput for each example, *xor_out* (remember that this time the training set is **not** linearly separable).

In [ ]:
xor_in = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
xor_out = np.array([[0], [1], [1], [0]])

Let us measure which is the error associated to the initial neural network before the training starts:

In [ ]:
print(my_net.sim(xor_in))
print(my_net.errorf(xor_out, my_net.sim(xor_in)))

Let us now proceed to run the training process on the neural network. The functions involved in the training work over the following arguments:
* `lr`: _learning rate_, default value 0.01.
* `epochs`: maximum number of epochs, default value 500.
* `show`: number of epochs that should be executed between two messages in the output log, default value 100.
* `goal`: maximum error accepted (halting criterion), default value 0.01.

In [ ]:
my_net.train(xor_in, xor_out, lr=0.1, epochs=50, show=10, goal=0.001)
my_net.sim(xor_in)

Let us now try a different setting. If we reset the neural network and we choose random numbers as initial values for the weights, we obtain the following:

In [ ]:
np.random.seed(3287426346)  # we set this init seed only for class, so that we always get
                               # the same random numbers and we can compare
my_net.reset()
for l in my_net.layers:
    l.initf = init.InitRand([-1, 1], 'bw')  # 'b' means biases will be modified,
                                            # and 'w' the weights
my_net.init()
my_net.train(xor_in, xor_out, lr=0.1, epochs=10000, show=1000, goal=0.001)
my_net.sim(xor_in)

# _Iris_ dataset

_Iris_ is a classic multivariant dataset that has been exhaustively studied and has become a standard reference when analysing the behaviour of different machine learning algorithms.

_Iris_ gathers four measurements (length and width of sepal and petal) of 50 flowers of each one of the following three species of lilies: _Iris setosa_, _Iris virginica_ and _Iris versicolor_.

Let us start by reading the data from the file `iris.csv` that has been provided together with the practice. It suffices to evaluate the following expressions:

In [ ]:
import pandas as pd

iris = pd.read_csv('iris.csv', header=None,
                   names=['Sepal length', 'sepal width',
                          'petal length', 'petal width',
                          'Species'])
iris.head(10)  # Display ten first examples

Next, let us move to use a numerical version of the species instead.<br>
Then, we should distribute the examples into two groups: training and test, and split each group into two components: input and expected output (goal).

In [ ]:
from sklearn import preprocessing
from sklearn import model_selection

iris_training, iris_test = model_selection.train_test_split(
   iris, test_size=.33, random_state=2346523,
   stratify=iris['Species'])

ohe = preprocessing.OneHotEncoder()
input_training = iris_training.iloc[:, :4]
goal_training = ohe.fit_transform(iris_training['Species'].values.reshape(-1, 1))
input_test = iris_test.iloc[:, :4]
goal_test = ohe.transform(iris_test['Species'].values.reshape(-1,1))

In [ ]:
goal_test.toarray()

__Exercise 1__: define a function **lily_species** that, given an array with three numbers as input, returns the position where the maximum value is.

In [ ]:
# Tip: np.argmax method

__Exercise 2__: Create a feed forward neural network having the following features:
1. Has four input neurons, one for each attribute of the iris dataset.
2. Has three output neurons, one for each species.
3. Has one hidden layer with two neurons.
4. All neurons of all layers use the sigmoid as activation function.
5. The initial biases and weights are all equal to zero.
6. Training method is gradient descent backpropagation.
7. The error function is the mean squared error.

Once you have created it, train the network over the sets `input_training` and `goal_training`.

In [ ]:
# Tip: obtain min and max values for each column
attrs = [c for c in iris_training.columns[:-1]]
iris_min_max = [[min(iris_training[c]), max(iris_training[c])] for c in attrs]
iris_min_max

__Exercise 3__: Calculate the performance of the network that was trained on the previous exercise, using to this aim the sets `input_test` and `goal_test`. That is, calculate which fraction of the test set is getting the correct classification predicted by the network.<br>
__Hint:__ In order to translate the output of the network and obtain which is the species predicted, use the function from exercise 1.

__Exercise 4__: try to create different variants of the network from exercise 2, by modifying the number of hidden layers and/or the amount of neurons per layer, in such a way that the performance over the test set is improved.

__Assignment 1__: Use any other of the datasets provided in the .zip file in moodle. Preprocess the data, split it into appropriate training and testing sets, and follow the process to choose the network structure from different variants. Is its performance better or worse than a decision tree and/or a k-nn?

The code below (in one or several cells) must provide enough evidence to support the answer to previous question. Include the neccesary comments to the provided code and the response to the question as well as its justification as comments or as a markdown cell.

__Note__: copying or just using AI tools to obtain the code without understanding it is not allowed. The professor reserves the right to make occasional defenses to the assignment deliveries, either random or to solve doubts about their authorship.

In [ ]:
# Solution
# ...
np.random.seed(3287426346)  # using the same seed used in class for the same reason

rice = pd.read_csv('P2_data/rice.csv') # headers are already defined in the file:
# Area,Perimeter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,Class
print(rice.head(10))  # Display ten first examples
# We can see that we have numeric values for every header except for the class

print("Classes:", rice['Class'].unique()) # We only have 2 outcomes ['Cammeo', 'Osmancik']

# We can use a lable encoder, because we only have 2 classes and use 1 ouput neuron
le = preprocessing.LabelEncoder()
rice['Class'] = le.fit_transform(rice['Class'])
print("Classes:", rice['Class'].unique()) # [0 1]

# Split train and test
rice_training, rice_test = model_selection.train_test_split(
   rice, test_size=.33, random_state=2346523,
   stratify=rice['Class']) # stratify to ensure % of the classes is similar in both splits

# Seperate input and goal
input_training = rice_training.iloc[:, :-1] # every row, every colum except the last Class
goal_training = rice_training.iloc[:, -1] # every row, only last colum Class

input_test = rice_test.iloc[:, :-1]
goal_test = rice_test.iloc[:, -1]

# Scaling feature because Area values are high compared to other values like Eccentricity
ss = preprocessing.StandardScaler()
input_training_scaled = ss.fit_transform(input_training)
input_test_scaled = ss.transform(input_test)

# Train the ANN
from sklearn import neural_network # could also use neurolab, but this is simpler
ann = neural_network.MLPClassifier(hidden_layer_sizes=(7,), random_state=2346523) # not changing default values like learning rate or iterations
# just setting random_state for reproducibility and trying out different hidden layer sizes to figure out a goog network structure
ann.fit(input_training_scaled, goal_training)

# Accuracy
from sklearn import metrics
input_test_scaled_predicted = ann.predict(input_test_scaled)
accuracy = metrics.accuracy_score(goal_test, input_test_scaled_predicted)
#print("ANN Accuracy:", accuracy)

print("Input layer size:", ann.n_features_in_)
print("Hidden layer size:", ann.hidden_layer_sizes)
print("Output layer size:", ann.n_outputs_)

# Func to determine the best hidden layer network structure based on example network structures
def find_best_network_structure(hidden_layers):
   best_network_structure = (0,) # init value 
   best_accuracy = -1 # init value

   for current_hidden_layer_sizes in hidden_layers:
      ann = neural_network.MLPClassifier(hidden_layer_sizes = current_hidden_layer_sizes, random_state=2346523)
      ann.fit(input_training_scaled, goal_training)
      input_test_scaled_predicted = ann.predict(input_test_scaled)
      accuracy = metrics.accuracy_score(goal_test, input_test_scaled_predicted)

      if accuracy > best_accuracy:
         best_accuracy = accuracy
         best_network_structure = current_hidden_layer_sizes

      #print("ANN Accuracy:", accuracy, "Hidden Layer Sizes:", current_hidden_layer_sizes)

   return best_network_structure, best_accuracy

# Just trying out single hidden layers because that is usually suffiecient for ANN to have just a single hidden layer
single_hidden_layers = [(1,), (2,), (3,), (5,), (10,), (20,), (50,)]
best_hidden_layer_network_structure, best_hidden_layer_network_structure_accuracy = find_best_network_structure(single_hidden_layers)
print("ANN Accuracy:" , best_hidden_layer_network_structure_accuracy,"Hidden Layer Sizes:", best_hidden_layer_network_structure ) # 0.921 , (10,)
# According to this small test (10,) seems to be the best network structure out of the tested structures for a single hidden layer

# Comparing performance to decision tree and/or a k-nn
from sklearn import tree
dt = tree.DecisionTreeClassifier(random_state=2346523)
dt.fit(input_training, goal_training) # scaled not required, because we are using a deciscion tree
y_pred_dt = dt.predict(input_test)
dt_acc = metrics.accuracy_score(goal_test, y_pred_dt)

print("Decision Tree Accuracy:", dt_acc) # 0.888

# K-NN
from sklearn import neighbors
knn = neighbors.KNeighborsClassifier()
knn.fit(input_training_scaled, goal_training)
y_pred_knn = knn.predict(input_test_scaled)
knn_acc = metrics.accuracy_score(goal_test, y_pred_knn)

print("K-NN Accuracy:", knn_acc) # 0.908

#print(f"According to my results the ANN ({round(best_hidden_layer_network_structure_accuracy,3)}) has a better Accuracy than the Decision Tree ({round(dt_acc,3)}) and the K-NN ({round(knn_acc,3)}).")

According to my results the ANN (0.921) has a better Accuracy than the Decision Tree (0.888) and the K-NN (0.908).